# BAAC road accidents: Île-de-France, one full pass per year

Every step below runs **once per year in `YEARS`**. Each year is downloaded from its own
files (`Caract_YYYY.csv`, `Lieux_YYYY.csv`, ...) and checked: `BAACDataset.from_data_gouv`
raises an error if a table does not actually contain accidents of the requested year.

Needs `baac.py`, `commune_analysis.py` and `commune_rankings.py` in the same folder.

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from IPython.display import display

from baac import BAACDataset, DataGouvLoader, TABLE_KEYS
from commune_analysis import build_idf_dataset, CommuneAggregator, fetch_idf_communes
from commune_rankings import CommuneRanker

logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")

# ---- configuration: change here only ----
YEARS = [2025, 2024, 2023, 2022]    # one pass of the whole notebook per year
MIN_ACCIDENTS = 30                  # communes below this are not ranked
TOP_N = 10
OUT_DIR = Path("data")              # tables, figures, csv
CACHE_DIR = Path("cache")           # raw CSVs: downloaded once, reused on later runs

OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load every year

Prints the files really used for each year, then the number of rows per table.

In [ ]:
loader = DataGouvLoader(cache_dir=CACHE_DIR)

national = {}                       # {year: BAACDataset for the whole country}
for year in YEARS:
    national[year] = BAACDataset.from_data_gouv(year, loader=loader)   # raises if not really `year`
    print(year, national[year].sources)

rows = pd.DataFrame({y: ds.summary()["rows"] for y, ds in national.items()}).T.astype("Int64")
rows.index.name = "year"
display(rows)

## 2. Do all years have the same columns?

Differences between a year and the latest year (empty table = identical schemas).

In [ ]:
latest = max(YEARS)
diffs = []
for year, ds in national.items():
    for key in TABLE_KEYS:
        here, there = key in ds.table_keys, key in national[latest].table_keys
        if here != there:
            diffs.append({"year": year, "table": key,
                          "only_in_year": "(whole table)" if here else "",
                          "only_in_latest": "(whole table)" if there else ""})
        elif here:
            a, b = set(ds.table(key).columns), set(national[latest].table(key).columns)
            if a != b:
                diffs.append({"year": year, "table": key,
                              "only_in_year": sorted(a - b), "only_in_latest": sorted(b - a)})

if diffs:
    display(pd.DataFrame(diffs))
else:
    print(f"Same tables and columns in every year ({min(YEARS)}-{max(YEARS)}).")

## 3. Schema figure for each year

Saved as `data/baac_<year>_schemas.png`; only the latest year is displayed.

In [ ]:
DTYPE_COLORS = {"str": "#fde4c8", "object": "#fde4c8", "int64": "#cfe3f7", "float64": "#d4efd2"}
KEY_COLOR, MISSING_COLOR = "#ffd95a", "#f5b5b5"
JOIN_KEYS = {"Num_Acc", "id_vehicule", "id_usager", "num_veh"}
HEADERS = ["column", "dtype", "non_null", "n_missing", "n_unique", "example"]


def describe_columns(df):
    return pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "non_null": df.notna().sum().values,
        "n_missing": df.isna().sum().values,
        "n_unique": df.nunique().values,
        "example": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })


def plot_schemas(ds, year):
    keys = [k for k in TABLE_KEYS if k in ds.table_keys]
    fig, axes = plt.subplots(2, 3, figsize=(24, 13))
    axes = axes.flatten()

    for ax, key in zip(axes, keys):
        df = ds.table(key)
        schema = describe_columns(df)
        ax.axis("off")
        ax.set_title(f"{ds.sources.get(key, key)}\n{len(df):,} rows × {len(schema)} columns",
                     fontsize=12, fontweight="bold", pad=8)

        cell_text = [[str(r.column), r.dtype, f"{r.non_null:,}", f"{r.n_missing:,}",
                      f"{r.n_unique:,}", str(r.example)[:18]] for r in schema.itertuples()]
        tbl = ax.table(cellText=cell_text, colLabels=HEADERS, loc="upper center", cellLoc="left",
                       colWidths=[0.27, 0.12, 0.14, 0.14, 0.14, 0.19])
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(9)
        tbl.scale(1, 1.55)

        for j in range(len(HEADERS)):
            tbl[0, j].set_facecolor("#2b2b2b")
            tbl[0, j].set_text_props(color="white", fontweight="bold")
        for i, r in enumerate(schema.itertuples(), start=1):
            tbl[i, 1].set_facecolor(DTYPE_COLORS.get(r.dtype, "#eeeeee"))
            if r.column in JOIN_KEYS:
                tbl[i, 0].set_facecolor(KEY_COLOR)
                tbl[i, 0].set_text_props(fontweight="bold")
            if r.n_missing > 0:
                tbl[i, 3].set_facecolor(MISSING_COLOR)

    for ax in axes[len(keys):]:
        ax.axis("off")
    axes[-1].legend(handles=[
        mpatches.Patch(color="#fde4c8", label="str / object"),
        mpatches.Patch(color="#cfe3f7", label="int64"),
        mpatches.Patch(color="#d4efd2", label="float64"),
        mpatches.Patch(color=KEY_COLOR, label="join key (Num_Acc, id_vehicule, ...)"),
        mpatches.Patch(color=MISSING_COLOR, label="column has missing values"),
    ], loc="center", fontsize=13, title="Legend", title_fontsize=14, frameon=False)

    fig.suptitle(f"BAAC {year}: schema of each file", fontsize=18, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    return fig


for year, ds in national.items():
    fig = plot_schemas(ds, year)
    fig.savefig(OUT_DIR / f"baac_{year}_schemas.png", dpi=200, bbox_inches="tight")
    if year == max(YEARS):
        plt.show()
    plt.close(fig)

## 4. Île-de-France pipeline for each year

For every year: keep the 8 Île-de-France departments, aggregate by commune (communes with
0 accidents included, using the official commune list), run the integrity checks, rank, and save
`data/top<N>_severity_<year>` and `data/top<N>_vulnerability_<year>` (`.png` and `.csv`).

In [ ]:
ref = fetch_idf_communes()          # official list of IDF communes (the same for every year)
if ref is None:
    print("Official commune list unavailable: communes with 0 accidents will not be listed.")

by_year, summary = {}, []
for year, ds in national.items():
    idf = build_idf_dataset(ds)
    by = CommuneAggregator(idf, reference_communes=ref).by_commune()

    # integrity checks, per year
    assert by["accidents"].sum() == len(idf.caract), f"{year}: accidents lost in the aggregation"
    assert by.index.is_unique, f"{year}: duplicated commune codes"

    CommuneRanker(by, year=year, min_accidents=MIN_ACCIDENTS, top_n=TOP_N).save(OUT_DIR)

    by_year[year] = by
    summary.append({
        "year": year,
        "accidents_france": len(ds.caract),
        "accidents_idf": len(idf.caract),
        "communes_with_accidents": int((by["accidents"] > 0).sum()),
        "communes_with_0": int((by["accidents"] == 0).sum()),
        "communes_ranked": int((by["accidents"] >= MIN_ACCIDENTS).sum()),
        "codes_not_in_reference": int((~by["in_reference"]).sum()) if "in_reference" in by else None,
    })
    print(f"{year}: done")

In [ ]:
summary = pd.DataFrame(summary).set_index("year")
display(summary)

**Reading the summary**

- Different numbers in every row confirm that each year comes from its own files (identical rows would be a red flag).
- `codes_not_in_reference`: commune codes present in the accident data but not in the *current* official list
  (Paris coded `75`, communes merged since, typos). These accidents are still counted.
- `communes_ranked`: how many communes pass the `MIN_ACCIDENTS` threshold and can appear in a top 10.

In [ ]:
year = max(YEARS)
by_year[year].head(15)

## 5. All years in one table

One row per (year, commune), saved for trend analysis.

In [ ]:
all_years = pd.concat(by_year, names=["year", "com"])
path = OUT_DIR / f"by_commune_{min(YEARS)}-{max(YEARS)}.csv"
all_years.to_csv(path)
print(f"{len(all_years):,} rows saved to {path}")
all_years.head()